# Подготовка данных и признаков для классических моделей

Ноутбук загружает обучающую и тестовую выборки, подготавливает признаки (word TF-IDF, char TF-IDF, числовые) и обучает базовые модели без настройки гиперпараметров.

Результаты сохраняются в pickle-файлы для использования в последующих ноутбуках:
- `03b_optuna_tuning.ipynb` — поиск гиперпараметров
- `03c_evaluation.ipynb` — оценка моделей
- `03d_ensembles_and_analysis.ipynb` — ансамбли и анализ

## Импорты

In [1]:
import os
import sys
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from scipy.sparse import hstack

try:
    _project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
except NameError:
    _cwd = os.getcwd()
    _project_root = _cwd
    while _project_root != '/' and not os.path.isdir(os.path.join(_project_root, 'src')):
        _project_root = os.path.dirname(_project_root)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from src.models.classical import (
    NUMERICAL_COLUMNS,
    prepare_features,
    train_classical_models,
    predict_message,
    save_models,
)
from src.evaluation.metrics import optimize_threshold
from src.config import PROCESSED_DIR, MODELS_DIR, INTERIM_DIR

import optuna
import lightgbm
import catboost
import nltk

nltk.download('stopwords', quiet=True)
warnings.filterwarnings('ignore', category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## Загрузка данных

Загружаются:
- `preprocessed.csv` — обучающая выборка (77K строк)
- `test_preprocessed.csv` — тестовая выборка (41K строк, подготовлена в `01d_test_preparation.ipynb`)

Тестовая выборка не пересекается с обучающей (data leakage устранён в `01b_data_cleaning.ipynb`).

In [2]:
df_train = pd.read_csv(PROCESSED_DIR / 'preprocessed.csv', index_col=0)
df_test = pd.read_csv(PROCESSED_DIR / 'test_preprocessed.csv')

print(f'Обучающая выборка: {len(df_train)} строк')
print(f"  Ham (0): {len(df_train[df_train['label'] == 0])}")
print(f"  Spam (1): {len(df_train[df_train['label'] == 1])}")
print(f'\nТестовая выборка: {len(df_test)} строк')
print(f"  Ham (0): {len(df_test[df_test['label'] == 0])}")
print(f"  Spam (1): {len(df_test[df_test['label'] == 1])}")

Обучающая выборка: 77523 строк
  Ham (0): 59086
  Spam (1): 18437

Тестовая выборка: 41369 строк
  Ham (0): 22625
  Spam (1): 18744


Проверка наличия всех числовых признаков в датасете.

In [3]:
missing_cols = [c for c in NUMERICAL_COLUMNS if c not in df_train.columns]
if missing_cols:
    raise ValueError(f'В датасете отсутствуют колонки: {missing_cols}')
print(f'Числовых признаков: {len(NUMERICAL_COLUMNS)}')
print(f'Колонки: {NUMERICAL_COLUMNS}')

Числовых признаков: 20
Колонки: ['emojis', 'newlines', 'whitespaces', 'links', 'tags', 'length', 'capital_ratio', 'punctuation_count', 'digit_count', 'avg_word_length', 'word_count', 'unique_word_ratio', 'repeat_char_ratio', 'phone_count', 'has_crypto', 'exclamation_count', 'url_ratio', 'html_tag_count', 'has_markdown', 'emoji_diversity']


## Подготовка признаков

Создаются три типа признаков:
1. **Word TF-IDF** — униграммы + биграммы, 5000 признаков, стоп-слова ru+en
2. **Char TF-IDF** — символьные n-граммы (2-5), 2000 признаков — для ловли обфусцированного спама (гомоглифы, leet speak, искажённые слова)
3. **Числовые признаки** — 20 колонок (эмодзи, ссылки, теги, длина текста и др.)

Признаки объединяются через `scipy.sparse.hstack`. Числовые признаки масштабируются `StandardScaler` (кроме версии для Naive Bayes — без масштабирования, так как NB требует неотрицательные значения).

Удаление строк с пустым `text_preprocessed` и подготовка признаков.

In [4]:
df_train = df_train.dropna(subset=['text_preprocessed']).reset_index(drop=True)
df_test = df_test.dropna(subset=['text_preprocessed']).reset_index(drop=True)

(
    X_train, X_test, y_train, y_test,
    vectorizer, char_vectorizer, scaler,
    X_train_nc, X_test_nc,
) = prepare_features(
    df_train, df_test,
    use_char_tfidf=True,
    random_state=RANDOM_STATE,
)

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'X_train_nc: {X_train_nc.shape}')
print(f'Word TF-IDF: {len(vectorizer.get_feature_names_out())} признаков')
print(f'Char TF-IDF: {len(char_vectorizer.get_feature_names_out())} признаков')
print(f'Числовых: {len(NUMERICAL_COLUMNS)} признаков')

X_train: (77523, 7020)
X_test:  (41369, 7020)
X_train_nc: (77523, 7020)
Word TF-IDF: 5000 признаков
Char TF-IDF: 2000 признаков
Числовых: 20 признаков


## Обучение базовых моделей

Обучаются 5 моделей со стандартными параметрами и `class_weight='balanced'`:
- Logistic Regression
- Random Forest
- Naive Bayes
- LightGBM
- CatBoost

Это бейзлайн для сравнения с моделями после Optuna.

In [5]:
base_models = train_classical_models(
    X_train, y_train,
    X_train_nc,
    random_state=RANDOM_STATE,
)
print(f'Обучено моделей: {len(base_models)}')
print(f'Модели: {list(base_models.keys())}')

Обучено моделей: 5
Модели: ['lr', 'rf', 'nb', 'lgbm', 'catboost']


## Оценка базовых моделей на тестовой выборке

Функция для вычисления метрик на тестовой выборке. Используется во всех последующих ноутбуках.

In [6]:
def evaluate_on_test(models, X_test, y_test, X_test_nc=None, prefix='base'):
    """Вычисляет метрики для каждой модели на тестовой выборке"""
    results = []
    for name, model in models.items():
        X_eval = X_test_nc if name == 'nb' and X_test_nc is not None else X_test
        y_pred = model.predict(X_eval)
        y_proba = model.predict_proba(X_eval)[:, 1]

        results.append({
            'model': f'{prefix}_{name}',
            'accuracy': float((y_pred == y_test).mean()),
            'precision_spam': float(precision_score(y_test, y_pred, pos_label=1, zero_division=0)),
            'recall_spam': float(recall_score(y_test, y_pred, pos_label=1, zero_division=0)),
            'f1_spam': float(f1_score(y_test, y_pred, pos_label=1, zero_division=0)),
            'f1_macro': float(f1_score(y_test, y_pred, average='macro', zero_division=0)),
            'roc_auc': float(roc_auc_score(y_test, y_proba)),
            'pr_auc': float(average_precision_score(y_test, y_proba)),
        })
    return pd.DataFrame(results)

Запуск оценки базовых моделей.

In [7]:
y_test_arr = np.array(y_test)
base_results = evaluate_on_test(base_models, X_test, y_test_arr, X_test_nc, prefix='base')
base_results

,model,accuracy,precision_spam,recall_spam,f1_spam,f1_macro,roc_auc,pr_auc
0,base_lr,0.960091,0.990417,0.920828,0.954356,0.959451,0.990388,0.990554
1,base_rf,0.956272,0.992497,0.910371,0.949662,0.955504,0.977944,0.982914
2,base_nb,0.856390,0.985809,0.693022,0.813884,0.848487,0.950470,0.955724
3,base_lgbm,0.954144,0.991596,0.906477,0.947128,0.953322,0.980168,0.983795
4,base_catboost,0.948996,0.991084,0.895487,0.940863,0.948012,0.971332,0.978399


## Сохранение признаков и моделей для последующих ноутбуков

Признаки, векторизаторы и базовые модели сохраняются в pickle-файлы в `data/interim/`.

In [8]:
with open(INTERIM_DIR / 'features_train.pkl', 'wb') as f:
    pickle.dump({'X_train': X_train, 'X_train_nc': X_train_nc, 'y_train': y_train}, f)

Сохранено:
  /home/sophrosyne/STANKIN_AntiSpam_Bot/data/interim/features_train.pkl
  /home/sophrosyne/STANKIN_AntiSpam_Bot/data/interim/features_test.pkl
  /home/sophrosyne/STANKIN_AntiSpam_Bot/data/interim/vectorizers.pkl
  /home/sophrosyne/STANKIN_AntiSpam_Bot/data/interim/base_models.pkl


In [9]:
with open(INTERIM_DIR / 'features_test.pkl', 'wb') as f:
    pickle.dump({'X_test': X_test, 'X_test_nc': X_test_nc, 'y_test': y_test}, f)

In [10]:
with open(INTERIM_DIR / 'vectorizers.pkl', 'wb') as f:
    pickle.dump({
        'vectorizer': vectorizer,
        'char_vectorizer': char_vectorizer,
        'scaler': scaler,
    }, f)

In [11]:
with open(INTERIM_DIR / 'base_models.pkl', 'wb') as f:
    pickle.dump(base_models, f)

In [12]:
print('Сохранено:')
print(f'  {INTERIM_DIR / "features_train.pkl"}')
print(f'  {INTERIM_DIR / "features_test.pkl"}')
print(f'  {INTERIM_DIR / "vectorizers.pkl"}')
print(f'  {INTERIM_DIR / "base_models.pkl"}')

Сохранено:
  /home/sophrosyne/STANKIN_AntiSpam_Bot/data/interim/features_train.pkl
  /home/sophrosyne/STANKIN_AntiSpam_Bot/data/interim/features_test.pkl
  /home/sophrosyne/STANKIN_AntiSpam_Bot/data/interim/vectorizers.pkl
  /home/sophrosyne/STANKIN_AntiSpam_Bot/data/interim/base_models.pkl
